## 非交互式蛋白质演化模型推理

此 Notebook 用于对单个蛋白质进行推理。请按照以下步骤操作：

1. **修改下方单元格中的 `PROTEIN_NAME`**：将其设置为您想要分析的蛋白质的名称（例如 `'S'`, `'NSP5'` 等）。
2. **按顺序运行所有单元格**：点击菜单栏的 `Kernel` -> `Restart & Run All`。
3. **查看输出**: 最终的预测结果会显示在最后一个单元格的输出区域。

In [29]:
import torch
import json
import os
import warnings

# 导入您项目中的工具函数和模块
import trainUtils
import ioutils
import modules
from esm.sdk.api import ESMProtein

# 忽略一些不影响结果的警告
warnings.filterwarnings("ignore")

# ===================================================================
# --- ✨ 用户配置区 ✨ ---
# 请在这里修改您想要预测的蛋白质名称
PROTEIN_NAME = "S"
# ===================================================================

print(f"配置完成，将对蛋白质 '{PROTEIN_NAME}' 进行预测。")

配置完成，将对蛋白质 'S' 进行预测。


In [30]:
print("--- 步骤 1: 准备环境和加载模型 ---")
# --- 配置路径和设备 ---
CHECKPOINT_DIR = "/data2/zhoukaitao/01evoModel/checkpoints/250904_local_7countries/"
CONFIG_FILE = os.path.join(CHECKPOINT_DIR, "config.json")
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, "last.ckpt")
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"将使用设备: {DEVICE}")

# --- 加载模型 ---
print(f"正在从 {CHECKPOINT_FILE} 加载模型...")
with open(CONFIG_FILE, "r") as f:
    configs = json.load(f)

pretrain_model = trainUtils.loadPretrainModel(configs)
model = trainUtils.buildModel(configs, pretrain_model, CHECKPOINT_FILE)

model.eval()
model.to(DEVICE)

print("✅ 模型加载成功并已设置为评估模式。")

--- 步骤 1: 准备环境和加载模型 ---
将使用设备: cuda:0
正在从 /data2/zhoukaitao/01evoModel/checkpoints/250904_local_7countries/last.ckpt 加载模型...
load local model: /data2/zhoukaitao/01evoModel/checkpoints/250813_stage1_test/pretrain_stage1_250824.pth
model at stage: training stage 1
load model from checkpoint /data2/zhoukaitao/01evoModel/checkpoints/250904_local_7countries/last.ckpt
✅ 模型加载成功并已设置为评估模式。


In [ ]:
print(f"\n--- 步骤 2: 为蛋白质 '{PROTEIN_NAME}' 准备输入数据 ---")

# --- 根据蛋白质名称自动生成文件路径 ---
fasta_path = f"fasta/predict/{PROTEIN_NAME}.fasta"
pdb_path = f"pdb/{PROTEIN_NAME}.pdb"

input_dict = None

try:
    # 1. 准备蛋白质对象
    #    我们总是先尝试从PDB加载，因为它包含最完整的信息
    if os.path.exists(pdb_path):
        print(f"找到 PDB 文件: {pdb_path}")
        protein_to_encode = ESMProtein.from_pdb(pdb_path)
        # 如果FASTA文件中的序列与PDB不同，可以选择覆盖PDB中的序列
        if os.path.exists(fasta_path):
            fna = ioutils.readFasta(fasta_path,remove='-toX')
            header, seq_from_fasta = next(fna)
            if protein_to_encode.sequence != seq_from_fasta:
                print("注意：FASTA序列与PDB序列不同，将使用FASTA序列进行预测。")
                protein_to_encode.sequence = seq_from_fasta
    elif os.path.exists(fasta_path):
        print(f"找到 FASTA 文件: {fasta_path} (无结构信息)")
        fna = ioutils.readFasta(fasta_path)
        header, seq_from_fasta = next(fna)
        protein_to_encode = ESMProtein(sequence=seq_from_fasta)
    else:
        raise FileNotFoundError(f"错误：找不到 {fasta_path} 或 {pdb_path}。无法进行预测。")
    
    # 2. 使用 model.encode() 获取所有对齐的Token
    with torch.no_grad():
        encoded_protein = pretrain_model.encode(protein_to_encode)
    
    # 3. 准备最终的输入字典
    input_dict = {
        PROTEIN_NAME: {
            'seq_t': encoded_protein.sequence.unsqueeze(0).to(DEVICE),
            'structure_t': encoded_protein.structure.unsqueeze(0).to(DEVICE) if encoded_protein.structure is not None else None,
            # 'structure_coords': protein_to_encode.coordinates.unsqueeze(0).to(DEVICE) if protein_to_encode.coordinates is not None else None
        }
    }
    print("✅ 输入数据准备完成。")

except Exception as e:
    print(f"❌ 在准备数据过程中发生错误: {e}")
    import traceback
    traceback.print_exc()


--- 步骤 2: 为蛋白质 'S' 准备输入数据 ---
找到 PDB 文件: pdb/S.pdb
注意：FASTA序列与PDB序列不同，将使用FASTA序列进行预测。
✅ 输入数据准备完成。


In [45]:
input_dict

{'S': {'seq_t': tensor([[ 0, 20, 18,  ..., 19, 11,  2]], device='cuda:0'),
  'structure_t': tensor([[4098, 3425, 1339,  ...,  490, 1272, 4097]], device='cuda:0')}}

In [46]:
print("\n--- 步骤 3: 执行模型推理并显示结果 ---")

if input_dict is not None:
    try:
        with torch.no_grad():
            model_output = model(input_dict)
        print("✅ 推理完成！")
        
        # --- 格式化并打印结果 ---
        print("\n" + "="*25 + " 模型输出结果 " + "="*25)
        if model_output.S1Embeddings:
            s1_embed = model_output.S1Embeddings[PROTEIN_NAME]
            print(f"  - Stage 1 Embedding (S1Embeddings):")
            print(f"    - 形状: {s1_embed.shape}")
            print(f"    - 均值: {s1_embed.mean().item():.4f}")
        
        if model_output.S1Predicts:
            s1_predict = model_output.S1Predicts[PROTEIN_NAME]
            print(f"\n  - Stage 1 预测 (S1Predicts):")
            for key, value in s1_predict.items():
                 print(f"    - {key}: {value.cpu().numpy()}")
        
        if model_output.S2Predicts:
            s2_predict = model_output.S2Predicts
            print(f"\n  - Stage 2 预测 (S2Predicts):")
            for key, value in s2_predict.items():
                 print(f"    - {key}: {value.cpu().numpy()}")
        print("="*68)

    except Exception as e:
        print(f"❌ 在模型推理过程中发生错误: {e}")
        import traceback
        traceback.print_exc()
else:
    print("输入数据未准备好，跳过推理。")


--- 步骤 3: 执行模型推理并显示结果 ---
✅ 推理完成！

========================= 模型输出结果 =========================
  - Stage 1 Embedding (S1Embeddings):
    - 形状: torch.Size([1, 1152])
    - 均值: 0.1340

  - Stage 1 预测 (S1Predicts):
    - predictions: []
    - time_series: [[-1.0126648]]


In [47]:
model_output.S1Predicts[PROTEIN_NAME]["time_series"][0]

tensor([-1.0127], device='cuda:0')